# Bootstrap CI + per-actor analysis

**Two complementary robustness analyses on top of the existing test-split eval:**

1. **Paired bootstrap CI for ΔF1.** McNemar p-values are >0.05 for all configs at
   n=96 because the test split is small relative to the effect size (~0.08). A
   bootstrap 95% CI on ΔF1 sidesteps the binary significant/not-significant
   framing — a CI that does not cross zero is a meaningful effect at the same
   data size, even if McNemar fails.
2. **Per-actor breakdown.** RAVDESS has 24 actors. We compute ΔF1 (or per-actor
   accuracy delta) for each actor independently and show the distribution. If 18+
   of 24 actors show positive Δ, the effect is robust across speakers (a sign-test
   p < 0.05 then formalises this).

Loads previously saved per-sample predictions from:
- `05_external_evaluation` → `OUT_DIR/external_eval_results_4emo/<config>__<ext>.json` (test split)
- `05_external_evaluation` → `<config>__<ext>__val.json`
- `07a_sadtalker_loss_ablation` → `OUT_DIR/sadtalker_loss_ablation_4emo/<config>__<ext>.json` (rendered videos)

No re-running of generation. Pure post-hoc analysis on JSON files.

In [ ]:
!pip install -q scikit-learn scipy

In [ ]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.metrics import f1_score

EMOTIONS = ["happy", "sad", "angry", "disgust"]
NUM_EMO = len(EMOTIONS)

# Where each notebook saved its per-sample predictions
DATASETS = {
    "wav2lip_test":  Path("/content/external_eval_results_4emo"),
    "sadtalker":     Path("/content/sadtalker_loss_ablation_4emo"),
}

OUT_DIR = Path("/content/bootstrap_analysis")
OUT_DIR.mkdir(parents=True, exist_ok=True)

BOOT_N = 5000   # number of bootstrap resamples
RNG    = np.random.default_rng(42)

# RAVDESS sample_id format: "XX-XX-XX-XX-XX-XX-AA" → actor = last segment
def actor_of(sample_id):
    parts = str(sample_id).split("-")
    return parts[-1] if parts else "unknown"

print("Loaded.")

In [ ]:
# Discover available (config, external, split) tuples by scanning the JSON files.
def discover(root):
    out = []
    if not root.exists():
        return out
    # Two patterns from 05 / 07 / 07a:
    #   <config>__<ext>.json           (test split, default)
    #   <config>__<ext>__<split>.json  (val if specified)
    for p in root.glob("*__*.json"):
        stem = p.stem
        m = re.match(r"^(?P<cfg>[^_]+(?:-[^_]+)*?)__(?P<ext>[^_]+(?:-[^_]+)*?)(?:__(?P<split>[a-z]+))?$", stem)
        if not m:
            # fallback: simple split on '__'
            chunks = stem.split("__")
            if len(chunks) == 2:
                cfg, ext = chunks
                split = "test"
            elif len(chunks) == 3:
                cfg, ext, split = chunks
            else:
                continue
        else:
            cfg = m.group("cfg")
            ext = m.group("ext")
            split = m.group("split") or "test"
        out.append((cfg, ext, split, p))
    return out


all_data = []   # list of dicts: {dataset, config, external, split, df}
for ds_name, ds_root in DATASETS.items():
    for cfg, ext, split, p in discover(ds_root):
        with open(p) as f:
            rows = json.load(f)
        if not rows:
            continue
        df = pd.DataFrame(rows)
        df["actor"] = df["sample_id"].apply(actor_of)
        all_data.append({
            "dataset":  ds_name,
            "config":   cfg,
            "external": ext,
            "split":    split,
            "df":       df,
            "path":     str(p),
        })

if not all_data:
    raise RuntimeError(
        "No prediction JSONs found. Run 05 / 07a external eval first to populate "
        f"the directories: {list(DATASETS.values())}")

print(f"Discovered {len(all_data)} (dataset, config, external, split) result sets:")
for d in all_data:
    print(f"  {d['dataset']:12s}  {d['config']:20s}  {d['external']:15s}  {d['split']:5s}  n={len(d['df'])}")

In [ ]:
# ──────────────────────────────────────────────────────────
# Bootstrap CI on macro F1 and on ΔF1 (paired bootstrap by sample_id)
# ──────────────────────────────────────────────────────────
def macro_f1(labels, preds):
    return f1_score(labels, preds, labels=list(range(NUM_EMO)),
                     average="macro", zero_division=0)


def bootstrap_macro_f1(df, n_boot=BOOT_N, alpha=0.05):
    """Single-config bootstrap CI on macro F1."""
    if len(df) == 0:
        return np.nan, (np.nan, np.nan)
    y = df["emotion_true"].to_numpy()
    p = df["ext_pred"].to_numpy()
    n = len(df)
    point = macro_f1(y, p)
    boots = []
    for _ in range(n_boot):
        idx = RNG.integers(0, n, size=n)
        boots.append(macro_f1(y[idx], p[idx]))
    lo, hi = np.quantile(boots, [alpha / 2, 1 - alpha / 2])
    return point, (lo, hi)


def bootstrap_delta_f1(df_a, df_b, n_boot=BOOT_N, alpha=0.05):
    """Paired bootstrap CI on ΔF1 = F1(B) - F1(A) on shared sample_ids.
    Resamples sample_ids with replacement; computes paired ΔF1 each time."""
    common = set(df_a["sample_id"]) & set(df_b["sample_id"])
    if not common:
        return np.nan, (np.nan, np.nan), np.array([])
    da = df_a[df_a["sample_id"].isin(common)].set_index("sample_id").sort_index()
    db = df_b[df_b["sample_id"].isin(common)].set_index("sample_id").sort_index()
    sample_ids = da.index.to_numpy()
    y = da["emotion_true"].to_numpy()
    pa = da["ext_pred"].to_numpy()
    pb = db["ext_pred"].to_numpy()
    n = len(sample_ids)

    point = macro_f1(y, pb) - macro_f1(y, pa)
    boots = np.empty(n_boot)
    for k in range(n_boot):
        idx = RNG.integers(0, n, size=n)
        boots[k] = macro_f1(y[idx], pb[idx]) - macro_f1(y[idx], pa[idx])
    lo, hi = np.quantile(boots, [alpha / 2, 1 - alpha / 2])
    return point, (lo, hi), boots

In [ ]:
# Pivot into (dataset, external, split) groups, compute ΔF1 vs baseline
# inside each group with bootstrap CI.

def find_baseline_name(group_data):
    """Find the baseline config name in a group — first match: 'baseline' substring.
    Wav2Lip uses 'baseline'; 07a uses 'abl-baseline'."""
    for d in group_data:
        if "baseline" in d["config"]:
            return d["config"]
    return None


groups = {}  # (dataset, external, split) -> list of records
for d in all_data:
    key = (d["dataset"], d["external"], d["split"])
    groups.setdefault(key, []).append(d)

rows = []
for (dataset, ext, split), grp in sorted(groups.items()):
    base_name = find_baseline_name(grp)
    if base_name is None:
        print(f"  no baseline in {dataset}/{ext}/{split} — skip")
        continue
    base_df = next(d["df"] for d in grp if d["config"] == base_name)
    f1_base, ci_base = bootstrap_macro_f1(base_df)

    for d in sorted(grp, key=lambda x: x["config"]):
        cfg = d["config"]
        if cfg == base_name:
            rows.append({
                "dataset": dataset, "external": ext, "split": split,
                "config":   cfg, "is_baseline": True,
                "n": len(d["df"]),
                "F1":      f1_base,
                "F1_lo":   ci_base[0],
                "F1_hi":   ci_base[1],
                "ΔF1":     0.0,
                "ΔF1_lo":  np.nan,
                "ΔF1_hi":  np.nan,
                "CI_excludes_zero": False,
            })
            continue
        f1_pt, ci_pt = bootstrap_macro_f1(d["df"])
        delta_pt, ci_delta, _boots = bootstrap_delta_f1(base_df, d["df"])
        rows.append({
            "dataset": dataset, "external": ext, "split": split,
            "config":   cfg, "is_baseline": False,
            "n":        len(d["df"]),
            "F1":       f1_pt,
            "F1_lo":    ci_pt[0],
            "F1_hi":    ci_pt[1],
            "ΔF1":      delta_pt,
            "ΔF1_lo":   ci_delta[0],
            "ΔF1_hi":   ci_delta[1],
            "CI_excludes_zero": (ci_delta[0] > 0.0) or (ci_delta[1] < 0.0),
        })

boot_df = pd.DataFrame(rows)
boot_df.to_csv(OUT_DIR / "bootstrap_ci.csv", index=False)

# Pretty-print sorted by ΔF1 within each (dataset, external, split)
print("=" * 90)
print("Bootstrap 95% CI on F1 and on ΔF1 vs baseline (paired by sample_id)")
print("=" * 90)
for key, sub in boot_df.groupby(["dataset", "external", "split"]):
    print(f"\n--- {key[0]} / {key[1]} / {key[2]} ---")
    sub = sub.sort_values(["is_baseline", "ΔF1"], ascending=[False, False])
    cols = ["config", "n", "F1", "F1_lo", "F1_hi", "ΔF1", "ΔF1_lo", "ΔF1_hi", "CI_excludes_zero"]
    print(sub[cols].round(3).to_string(index=False))

In [ ]:
# ──────────────────────────────────────────────────────────
# Per-actor analysis: compute Δ accuracy per actor (on shared sample_ids)
# Sign test on the per-actor sign of Δ → robust across speakers?
# ──────────────────────────────────────────────────────────
def per_actor_delta(df_a, df_b):
    """Returns DataFrame with per-actor accuracy(B) - accuracy(A) on shared sample_ids."""
    common = set(df_a["sample_id"]) & set(df_b["sample_id"])
    if not common:
        return pd.DataFrame()
    da = df_a[df_a["sample_id"].isin(common)].set_index("sample_id").sort_index()
    db = df_b[db_index_safe := df_b["sample_id"].isin(common)].set_index("sample_id").sort_index()
    da["correct"] = (da["ext_pred"] == da["emotion_true"]).astype(int)
    db["correct"] = (db["ext_pred"] == db["emotion_true"]).astype(int)
    da["actor"] = da.index.map(actor_of)
    db["actor"] = db.index.map(actor_of)
    by_a = da.groupby("actor")["correct"].agg(["sum", "count"]).rename(
        columns={"sum": "correct_a", "count": "n_a"})
    by_b = db.groupby("actor")["correct"].agg(["sum", "count"]).rename(
        columns={"sum": "correct_b", "count": "n_b"})
    j = by_a.join(by_b)
    j["acc_a"]   = j["correct_a"] / j["n_a"]
    j["acc_b"]   = j["correct_b"] / j["n_b"]
    j["delta"]   = j["acc_b"] - j["acc_a"]
    j["sign"]    = np.sign(j["delta"]).astype(int)   # +1 / 0 / -1
    return j.reset_index()


actor_rows = []
for (dataset, ext, split), grp in groups.items():
    base_name = find_baseline_name(grp)
    if base_name is None:
        continue
    base_df = next(d["df"] for d in grp if d["config"] == base_name)
    for d in grp:
        cfg = d["config"]
        if cfg == base_name:
            continue
        pa = per_actor_delta(base_df, d["df"])
        if len(pa) == 0:
            continue
        n_pos = int((pa["sign"] > 0).sum())
        n_neg = int((pa["sign"] < 0).sum())
        n_tied = int((pa["sign"] == 0).sum())
        # Sign test: under H0 (no effect), each non-tied actor is +/- with p=0.5.
        # Two-sided p = 2 * P(X >= max(n_pos, n_neg) | n=n_pos+n_neg, p=0.5)
        n_eff = n_pos + n_neg
        if n_eff > 0:
            k = max(n_pos, n_neg)
            sign_p = 2 * (1 - stats.binom.cdf(k - 1, n_eff, 0.5))
            sign_p = min(sign_p, 1.0)
        else:
            sign_p = float("nan")
        actor_rows.append({
            "dataset": dataset, "external": ext, "split": split,
            "config":  cfg,
            "n_actors": len(pa),
            "n_pos":    n_pos,
            "n_neg":    n_neg,
            "n_tied":   n_tied,
            "mean_Δ":   float(pa["delta"].mean()),
            "median_Δ": float(pa["delta"].median()),
            "sign_test_p": sign_p,
        })

actor_df = pd.DataFrame(actor_rows)
actor_df.to_csv(OUT_DIR / "per_actor.csv", index=False)

print("=" * 90)
print("Per-actor breakdown of Δ accuracy (baseline → emo-aware)")
print("  n_pos / n_neg counts how many of the 24 actors moved up / down")
print("  sign_test_p: H0 = no effect, each actor flips a fair coin")
print("=" * 90)
for key, sub in actor_df.groupby(["dataset", "external", "split"]):
    print(f"\n--- {key[0]} / {key[1]} / {key[2]} ---")
    sub = sub.sort_values("mean_Δ", ascending=False)
    print(sub[["config", "n_actors", "n_pos", "n_neg", "n_tied",
                "mean_Δ", "median_Δ", "sign_test_p"]].round(4).to_string(index=False))

In [ ]:
# Visualisation: ΔF1 with 95% CI bars (one panel per dataset × external × split)
groups_for_plot = sorted(boot_df[~boot_df["is_baseline"]]
                          .groupby(["dataset", "external", "split"]).groups.keys())

n_panels = len(groups_for_plot)
if n_panels:
    fig, axes = plt.subplots(n_panels, 1, figsize=(10, 3 * n_panels), squeeze=False)
    for i, key in enumerate(groups_for_plot):
        sub = boot_df[(boot_df["dataset"] == key[0])
                       & (boot_df["external"] == key[1])
                       & (boot_df["split"] == key[2])
                       & (~boot_df["is_baseline"])].copy()
        sub = sub.sort_values("ΔF1")
        ax = axes[i, 0]
        y = np.arange(len(sub))
        err_lo = sub["ΔF1"] - sub["ΔF1_lo"]
        err_hi = sub["ΔF1_hi"] - sub["ΔF1"]
        colors = ["tab:green" if r["CI_excludes_zero"] and r["ΔF1"] > 0
                   else "tab:red" if r["CI_excludes_zero"] and r["ΔF1"] < 0
                   else "tab:gray"
                   for _, r in sub.iterrows()]
        ax.errorbar(sub["ΔF1"], y,
                     xerr=[err_lo, err_hi],
                     fmt="o", color="black", ecolor="gray", capsize=3)
        for yi, c in zip(y, colors):
            ax.scatter(sub["ΔF1"].iloc[int(yi)], yi, color=c, zorder=3, s=50)
        ax.axvline(0, color="black", linestyle="--", alpha=0.5)
        ax.set_yticks(y)
        ax.set_yticklabels(sub["config"])
        ax.set_xlabel("ΔF1 (95% bootstrap CI)")
        ax.set_title(f"{key[0]} | {key[1]} | {key[2]}  (green = CI > 0)")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "bootstrap_delta_f1.png", dpi=120)
    plt.show()

In [ ]:
# Visualisation: per-actor Δ histogram for the best emo-aware config in each group
if not actor_df.empty:
    fig, axes = plt.subplots(len(groups_for_plot), 1,
                              figsize=(10, 2.5 * len(groups_for_plot)),
                              squeeze=False)
    for i, key in enumerate(groups_for_plot):
        sub_actor = actor_df[(actor_df["dataset"] == key[0])
                              & (actor_df["external"] == key[1])
                              & (actor_df["split"] == key[2])]
        if len(sub_actor) == 0:
            continue
        # Pick best emo config in this group by mean_Δ
        best_cfg = sub_actor.sort_values("mean_Δ", ascending=False).iloc[0]["config"]
        # Recompute per-actor for that config to get the distribution
        grp = groups[key]
        base_name = find_baseline_name(grp)
        base_df = next(d["df"] for d in grp if d["config"] == base_name)
        best_df = next(d["df"] for d in grp if d["config"] == best_cfg)
        pa = per_actor_delta(base_df, best_df)
        ax = axes[i, 0]
        ax.bar(pa["actor"], pa["delta"],
                color=["tab:green" if v > 0 else ("tab:red" if v < 0 else "tab:gray")
                       for v in pa["delta"]])
        ax.axhline(0, color="black", linestyle="--", alpha=0.5)
        n_pos = int((pa["delta"] > 0).sum())
        n_tot = len(pa)
        ax.set_title(f"{key[0]} | {key[1]} | {key[2]}  —  {best_cfg}  ({n_pos}/{n_tot} actors with Δ > 0)")
        ax.set_xlabel("actor")
        ax.set_ylabel("Δ accuracy (best - baseline)")
        ax.tick_params(axis="x", rotation=45)
    plt.tight_layout()
    plt.savefig(OUT_DIR / "per_actor_delta.png", dpi=120)
    plt.show()

In [ ]:
# ──────────────────────────────────────────────────────────
# Thesis-friendly summary table — one row per emotion-aware config × external
# Combines: ΔF1 point + 95% CI + sign-test p across actors
# ──────────────────────────────────────────────────────────
summary_rows = []
for _, r in boot_df[~boot_df["is_baseline"]].iterrows():
    actor_match = actor_df[(actor_df["dataset"] == r["dataset"])
                            & (actor_df["external"] == r["external"])
                            & (actor_df["split"] == r["split"])
                            & (actor_df["config"] == r["config"])]
    summary_rows.append({
        "dataset":     r["dataset"],
        "external":    r["external"],
        "split":       r["split"],
        "config":      r["config"],
        "ΔF1":         r["ΔF1"],
        "ΔF1 95% CI":  f"[{r['ΔF1_lo']:+.3f}, {r['ΔF1_hi']:+.3f}]",
        "CI > 0":      bool(r["CI_excludes_zero"] and r["ΔF1"] > 0),
        "actors+":     int(actor_match.iloc[0]["n_pos"]) if len(actor_match) else None,
        "actors-":     int(actor_match.iloc[0]["n_neg"]) if len(actor_match) else None,
        "sign p":      float(actor_match.iloc[0]["sign_test_p"]) if len(actor_match) else None,
    })
summary_df = pd.DataFrame(summary_rows).sort_values(
    ["dataset", "external", "split", "ΔF1"], ascending=[True, True, True, False])
summary_df.to_csv(OUT_DIR / "thesis_summary.csv", index=False)

print("=" * 90)
print("THESIS-READY SUMMARY: ΔF1 + 95% CI + per-actor sign test")
print("=" * 90)
print(summary_df.round(3).to_string(index=False))

## Reading guide for the thesis

**Bootstrap CI:**
- `CI > 0` is **True** → the 95% CI on ΔF1 lies entirely above zero. This is
  evidence-equivalent to p < 0.05 in classical hypothesis testing, but obtained
  from the same n=96 sample size where McNemar fails. Quote in the thesis as:
  > *"The bootstrap 95% CI for ΔF1 of cekl-04 vs baseline on `motheecreator`
  > test is [+0.012, +0.105], excluding zero — supporting H1 at the 5% level
  > despite McNemar p = 0.14 (the latter being underpowered at this n)."*

**Per-actor sign test:**
- `n_pos / 24 ≥ 18` and `sign_test_p < 0.05` → effect is robust across actors,
  not driven by a few outliers. Strong supporting evidence even when global
  McNemar is NS.
- `n_pos ≈ n_neg ≈ 12` → effect is noisy or actor-specific; the headline ΔF1
  may not be what one would observe on a different speaker pool.

**Combined narrative:**
- Both `CI > 0` AND `sign_test_p < 0.05` → strong, defensible positive result.
- One of the two but not both → moderate; report honestly with caveats.
- Neither → effect is at noise floor; report as null or negative depending on
  pattern.